# StrikeVision · Reconocimiento de acciones

Laboratorio reproducible para clasificar técnicas por peleador. Las celdas de entrenamiento son explícitas: abrir este notebook no extrae poses, descarga pesos ni inicia un entrenamiento. `ambiguous` se conserva en auditoría y nunca se convierte en clase ni en negativo.

In [ ]:
from pathlib import Path
import json

ROOT = Path.cwd().resolve()
if not (ROOT / 'configs').is_dir():
    ROOT = ROOT.parents[1]
DATASET = ROOT / 'data/processed/datasets/action_recognition_v1'
ROOT, DATASET

## 1. Construir el manifiesto derivado

Ejecuta esta celda después de modificar anotaciones. Valida hashes, fragmentos terminados, IDs, correcciones y particiones por pelea. Solo escribe archivos derivados bajo `data/processed/`.

In [ ]:
# Descomenta para regenerar el manifiesto; no entrena.
# !python -s scripts/ml/run_action_recognition.py prepare

In [ ]:
report = json.loads((DATASET / 'report.json').read_text(encoding='utf-8'))
{key: report[key] for key in ['rounds', 'fights', 'audit_action_counts', 'excluded_ambiguous', 'excluded_unknown_attacker', 'test_fights', 'development_folds']}

## 2. Baselines y modelos temporales

Cada ejecución elige un fold de desarrollo, extrae únicamente las poses necesarias y limita negativos por pelea. No abrir el test para escoger hiperparámetros. Los comandos generan un candidato y un `report.json` bajo `outputs/action-recognition/<run>`.

In [ ]:
# Ejecuta uno por vez cuando quieras entrenar:
# !python -s scripts/ml/run_action_recognition.py baseline --model logistic_regression --fold 0 --run baseline-logreg-fold0
# !python -s scripts/ml/run_action_recognition.py baseline --model extra_trees --fold 0 --run baseline-trees-fold0
# !python -s scripts/ml/run_action_recognition.py temporal --model tcn --fold 0 --run tcn-fold0
# !python -s scripts/ml/run_action_recognition.py temporal --model stgcn_compact --fold 0 --run stgcn-fold0
# !python -s scripts/ml/run_action_recognition.py rgb --model x3d_linear_probe --fold 0 --run x3d-fold0
# !python -s scripts/ml/summarize_action_tournament.py baseline-logreg-fold0 baseline-trees-fold0 tcn-fold0 stgcn-fold0

## 3. Lectura de resultados

Compara `primary_macro_f1`, F1/precision/recall por técnica, soporte, cobertura de pose y coste. `other_strike` solo tiene 11 registros: reportarlo separado. No usar esta tabla para declarar conteo exacto de golpes conectados.

In [ ]:
run = ROOT / 'outputs/action-recognition/tcn-fold0/report.json'
if run.is_file():
    result = json.loads(run.read_text(encoding='utf-8'))
    display(result['metrics'])
else:
    print('Aún no existe ese run. Cambia la ruta después de entrenar manualmente.')

## 4. Gradio

Con un candidato entrenado, inicia el probador y pega el directorio del run. La UI procesa únicamente los frames solicitados y conserva el FPS del video.

In [ ]:
# !python -s scripts/ml/run_action_inference_app.py